In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import LinearSegmentedColormap
from rdkit import Chem
from rdkit.Chem import Draw
import random
import os
import io
from PIL import Image

# --- 1. Load Data ---
generic_f_year = pd.read_csv("/DISCO1/Aylin/Chemoinformatics/Chemical_Space/scaffolds.csv")

# --- 2. Setup Custom Palette ---
custom_colors = ["#d2c6e8", "#4246aa", "#030a5d"]
chem_cmap = LinearSegmentedColormap.from_list("chem_space", custom_colors)

def add_mol_annotation(ax, x, y, smiles, center_x, center_y, zoom=0.4):
    """
    x, y: Actual coordinates of the t-SNE point.
    center_x, center_y: Coordinates of the cell center where the image will sit.
    """
    if pd.isna(smiles) or smiles == "":
        return
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        # High-quality Cairo rendering
        d = Draw.MolDraw2DCairo(300, 300)
        opts = d.drawOptions()
        opts.bondLineWidth = 5            # Thick lines for visibility
        opts.clearBackground = False      # Transparent background start
        
        Draw.rdMolDraw2D.PrepareAndDrawMolecule(d, mol)
        d.FinishDrawing()
        
        img = Image.open(io.BytesIO(d.GetDrawingText())).convert("RGBA")
        
        # Manual Alpha-keying for white backgrounds
        datas = img.getdata()
        newData = []
        for item in datas:
            if item[0] > 240 and item[1] > 240 and item[2] > 240:
                newData.append((255, 255, 255, 0))
            else:
                newData.append(item)
        img.putdata(newData)

        imagebox = OffsetImage(img, zoom=zoom)
        ab = AnnotationBbox(
            imagebox, (x, y),          # The arrow points HERE (the actual scaffold point)
            xybox=(center_x, center_y), # The molecule sits HERE (the cell center)
            xycoords='data',
            boxcoords='data',           # Tell Matplotlib to use map coordinates for the box
            pad=0.5,
            frameon=True,
            bboxprops=dict(
                edgecolor='grey',  # Change to your desired hex code
                facecolor='white',    # Background of the small box
                alpha=0.4,            # Transparency of the background/frame
                linewidth=1.5        # Thickness of the frame
            ),
            box_alignment=(0.5, 0.5),
            arrowprops=dict(
                arrowstyle="-",
                color='grey',
                lw=1.5,
                alpha=0.4,
                shrinkA=0,  # Gap between arrow and point
                shrinkB=5  # Gap between arrow and molecule center
            )
        )
        ax.add_artist(ab)

# --- 3. Main Loop ---
output_dir = "/DISCO1/Aylin/Chemoinformatics/Chemical_Space/Scaffolds/"
os.makedirs(output_dir, exist_ok=True)

# Define Grid boundaries globally for the 4x4
bins = np.linspace(-150, 150, 5) 
bin_centers = (bins[:-1] + bins[1:]) / 2  

for current_year in range(1976, 2025):
    # Filter and Process Data
    df_year = generic_f_year[generic_f_year['year'] <= current_year].copy()
    df_year['freq'] = df_year.groupby('generic_framework')['n'].transform('sum')
    df_plot = df_year.drop_duplicates(subset=['generic_framework']).copy()
    
    # Assign points to 4x4 grid cells
    df_plot['grid_x_idx'] = pd.cut(df_plot['tsne1'], bins=bins, labels=[0, 1, 2, 3], include_lowest=True)
    df_plot['grid_y_idx'] = pd.cut(df_plot['tsne2'], bins=bins, labels=[0, 1, 2, 3], include_lowest=True)

    # Create Plot
    fig, ax = plt.subplots(figsize=(15, 11.7), facecolor='white')
    
    # Scatter plot background
    scatter = ax.scatter(
        df_plot['tsne1'], df_plot['tsne2'],
        s=np.log10(df_plot['freq'] + 1) * 50, 
        c=np.log10(df_plot['n'] + 1),
        cmap=chem_cmap,
        alpha=0.6,
        vmin=0, vmax=4.8165,
        edgecolors='none'
    )

    # 4x4 Grid Logic: Find most freq image per cell and place at center
    for gx in range(4):
        for gy in range(4):
            cx, cy = bin_centers[gx], bin_centers[gy]
            cell_data = df_plot[(df_plot['grid_x_idx'] == gx) & (df_plot['grid_y_idx'] == gy)]
            
            if not cell_data.empty:
                # Find the top scaffold
                most_freq_row = cell_data.nlargest(1, 'freq').iloc[0]
                
                # Draw with a leader line:
                # x, y = the actual dot on the plot
                # center_x, center_y = the middle of the 4x4 square
                add_mol_annotation(
                    ax, 
                    x=most_freq_row['tsne1'], 
                    y=most_freq_row['tsne2'], 
                    smiles=most_freq_row['generic_framework'],
                    center_x=cx,
                    center_y=cy,
                    zoom=0.3
                )

    # Formatting
    ax.set_xlim(-160, 160)
    ax.set_ylim(-160, 160)
    ax.text(0, 145, f"{current_year}", fontsize=25, ha='center', fontweight='bold', color='black')
    ax.set_axis_off()
    
    # Save
    plt.savefig(f"{output_dir}CS_{current_year}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Finished year: {current_year}")
    
    # to compile the video, run in terminal:
    # ffmpeg -framerate 5 -start_number 1976 -i CS_%d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -pix_fmt yuv420p -crf 18 Chemical_Space_Evolution.mp4

/tmp/ipykernel_3228303/1970045640.py:41: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  datas = img.getdata()


Finished year: 1976


/tmp/ipykernel_3228303/1970045640.py:41: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  datas = img.getdata()


Finished year: 1977


/tmp/ipykernel_3228303/1970045640.py:41: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  datas = img.getdata()


...

Finished year: 2022


/tmp/ipykernel_3228303/1970045640.py:41: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  datas = img.getdata()


Finished year: 2023


/tmp/ipykernel_3228303/1970045640.py:41: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  datas = img.getdata()


Finished year: 2024
